# 🧠 Notebook 09 — XAI (Explainable AI) — SHAP + Feature Importance

**The 'Interpretable' part of your project title**

This notebook answers: **WHY does the model predict Benign or Malignant?**

We use:
1. **SHAP** (SHapley Additive exPlanations) — industry gold standard for XAI
2. **Feature Importance** from Random Forest
3. **Individual prediction explanation** — what pushed a specific image's prediction

**Run each cell with Shift+Enter**

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, r'D:\Projects\ML')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from src.config import (FEATURES_FUSED_CSV, FIGURES_DIR,
                        MODELS_DIR, RESULTS_DIR, RANDOM_SEED)
from src.utils import set_plot_style

set_plot_style()
np.random.seed(RANDOM_SEED)
print('SHAP version:', shap.__version__)
print('Imports done!')

## 1. Load Data & Best Model

In [ ]:
df = pd.read_csv(FEATURES_FUSED_CSV)
feat_cols = [c for c in df.columns if c.startswith('glcm_') or c.startswith('lbp_')]

test_df = df[df['split'] == 'test'].copy()
X_test  = test_df[feat_cols].values.astype(np.float32)
y_test  = test_df['label'].values

# Load Random Forest (best for SHAP TreeExplainer)
rf_model = joblib.load(MODELS_DIR / 'random_forest.pkl')

print(f'Test images: {len(X_test)}')
print(f'Features   : {len(feat_cols)}')
print(f'RF model loaded: {type(rf_model.named_steps["clf"]).__name__}')

## 2. Random Forest Feature Importance

In [ ]:
# Get the RF classifier and the selected features from SelectKBest
selector = rf_model.named_steps['select']
rf_clf   = rf_model.named_steps['clf']

# Get selected feature names
selected_mask  = selector.get_support()
selected_feats = [feat_cols[i] for i, m in enumerate(selected_mask) if m]

importances = rf_clf.feature_importances_
feat_imp_df = pd.DataFrame({'feature': selected_feats, 'importance': importances})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=False)

print('Top 15 Most Important Features (Random Forest):')
print(feat_imp_df.head(15).to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
top15 = feat_imp_df.head(15)
colors_imp = ['#7c3aed' if 'glcm' in f else '#059669' for f in top15['feature']]
bars = ax.barh(range(len(top15)), top15['importance'],
               color=colors_imp, edgecolor='#3d3f4e')
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15['feature'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Importance Score')
ax.set_title('Top 15 Feature Importances — Random Forest', fontsize=13)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#7c3aed', label='GLCM features'),
    Patch(facecolor='#059669', label='LBP features'),
]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / '09_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. SHAP Analysis — TreeExplainer on Random Forest

In [ ]:
# Transform test data through scaler + selector (before RF)
X_scaled   = rf_model.named_steps['scaler'].transform(X_test)
X_selected = selector.transform(X_scaled)

# Use sample of 200 for SHAP speed
sample_idx = np.random.choice(len(X_selected), min(200, len(X_selected)), replace=False)
X_shap     = X_selected[sample_idx]
y_shap     = y_test[sample_idx]

print('Computing SHAP values...')
explainer   = shap.TreeExplainer(rf_clf)
shap_values = explainer.shap_values(X_shap)

# shap_values shape: (n_samples, n_features, n_classes) or list
if isinstance(shap_values, list):
    sv_malignant = shap_values[1]  # class 1 = malignant
else:
    sv_malignant = shap_values[:, :, 1]

print(f'SHAP values shape: {sv_malignant.shape}')
print('Done!')

## 4. SHAP Summary Plot (Bar)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    sv_malignant, X_shap,
    feature_names=selected_feats,
    plot_type='bar',
    show=False, max_display=15
)
plt.title('SHAP Feature Importance (Malignant class)', fontsize=13)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / '09_shap_bar.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. SHAP Beeswarm Plot (Publication Quality)

In [ ]:
shap.summary_plot(
    sv_malignant, X_shap,
    feature_names=selected_feats,
    show=False, max_display=15
)
plt.title('SHAP Beeswarm — Impact on Malignant Prediction', fontsize=13)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / '09_shap_beeswarm.png'), dpi=150, bbox_inches='tight')
plt.show()

print('How to read: Red = high feature value, Blue = low feature value')
print('Points on right = pushed prediction toward Malignant')
print('Points on left  = pushed prediction toward Benign')

## 6. Individual Prediction Explanation

In [ ]:
# Explain one benign and one malignant image prediction
benign_idx    = np.where(y_shap == 0)[0][0]
malignant_idx = np.where(y_shap == 1)[0][0]

for idx, label in [(benign_idx, 'Benign'), (malignant_idx, 'Malignant')]:
    print(f'\n--- Individual Explanation: {label} image ---')
    top_feats = np.argsort(np.abs(sv_malignant[idx]))[::-1][:5]
    for rank, fi in enumerate(top_feats, 1):
        direction = 'toward Malignant' if sv_malignant[idx, fi] > 0 else 'toward Benign'
        print(f'  {rank}. {selected_feats[fi]:30s}: SHAP={sv_malignant[idx,fi]:+.4f} ({direction})')

## 7. Summary

In [ ]:
top5 = feat_imp_df.head(5)['feature'].tolist()
print('='*60)
print('  NOTEBOOK 09 COMPLETE — Interpretable AI')
print('='*60)
print(f'  Top 5 most important features:')
for i, f in enumerate(top5, 1):
    print(f'    {i}. {f}')
print()
print('  SHAP plots saved:')
print('  - 09_feature_importance.png')
print('  - 09_shap_bar.png')
print('  - 09_shap_beeswarm.png')
print()
print('  This is the Interpretable ML part of your title!')
print()
print('  Next: Streamlit App (Demo)')
print('='*60)